# 03 lag 实验

## 本文件目标
验证历史销量特征（lag）对零售销量预测的提升效果。

## 本文件主要工作
1. 解释 lag 特征的含义
2. 构造 train + test 的中间表生成 lag
3. 先尝试短 lag（lag_7 / lag_14）
4. 发现短 lag 在当前验证方式下可能偏乐观
5. 改为更安全的 lag_16、lag_21、lag_28
6. 重新训练并比较结果

## 核心理解
- lag 表示过去某一天的销量作为今天的特征
- lag 必须按 `store_nbr + family` 分组后生成
- 短 lag 可能读到验证窗口内部真实销量，导致分数偏乐观
- safe lag 更接近真实预测场景

In [ ]:
import pandas as pd
import numpy as np
from catboost import CatBoostRegressor

PATH = "/kaggle/input/competitions/store-sales-time-series-forecasting/"

train = pd.read_csv(PATH + "train.csv", parse_dates=["date"])
test = pd.read_csv(PATH + "test.csv", parse_dates=["date"])
stores = pd.read_csv(PATH + "stores.csv")
oil = pd.read_csv(PATH + "oil.csv", parse_dates=["date"])
holidays = pd.read_csv(PATH + "holidays_events.csv", parse_dates=["date"])
transactions = pd.read_csv(PATH + "transactions.csv", parse_dates=["date"])
sample_submission = pd.read_csv(PATH + "sample_submission.csv")

In [ ]:
base_train = train.merge(stores, on="store_nbr", how="left")
base_test = test.merge(stores, on="store_nbr", how="left")

base_train = base_train.merge(oil, on="date", how="left")
base_test = base_test.merge(oil, on="date", how="left")

base_train = base_train.sort_values("date").copy()
base_test = base_test.sort_values("date").copy()

base_train["dcoilwtico"] = base_train["dcoilwtico"].ffill().bfill()
base_test["dcoilwtico"] = base_test["dcoilwtico"].ffill().bfill()

In [ ]:
train_df = base_train.copy()
test_df = base_test.copy()

for df in [train_df, test_df]:
    df["year"] = df["date"].dt.year
    df["month"] = df["date"].dt.month
    df["day"] = df["date"].dt.day
    df["dayofweek"] = df["date"].dt.dayofweek
    df["dayofyear"] = df["date"].dt.dayofyear
    df["is_weekend"] = (df["dayofweek"] >= 5).astype(int)

cat_cols = ["store_nbr", "family", "city", "state", "type", "cluster"]
num_cols = ["onpromotion", "dcoilwtico", "year", "month", "day", "dayofweek", "dayofyear", "is_weekend"]
feature_cols = cat_cols + num_cols
target_col = "sales"

for col in cat_cols:
    train_df[col] = train_df[col].astype(str)
    test_df[col] = test_df[col].astype(str)

## 短 lag 的结论（不作为当前最佳版本）
- experiment_name: `lag_7_14_catboost_recent365`
- score: `0.444306`
- 说明：结果很强，但验证可能偏乐观，因为短 lag 可能使用验证窗口内部真实销量。

因此本 notebook 重点保留 **safe lag** 主版本。

In [ ]:
lag_df = pd.concat([
    train_df[["id", "date", "store_nbr", "family", "sales"]].copy(),
    test_df[["id", "date", "store_nbr", "family"]].assign(sales=np.nan).copy()
], ignore_index=True)

lag_df = lag_df.sort_values(["store_nbr", "family", "date"]).copy()

lag_df["lag_16"] = lag_df.groupby(["store_nbr", "family"])["sales"].shift(16)
lag_df["lag_21"] = lag_df.groupby(["store_nbr", "family"])["sales"].shift(21)
lag_df["lag_28"] = lag_df.groupby(["store_nbr", "family"])["sales"].shift(28)

train_df_lag = train_df.merge(
    lag_df[["id", "lag_16", "lag_21", "lag_28"]],
    on="id",
    how="left"
)

test_df_lag = test_df.merge(
    lag_df[["id", "lag_16", "lag_21", "lag_28"]],
    on="id",
    how="left"
)

print(train_df_lag[["lag_16", "lag_21", "lag_28"]].isna().sum())
print(test_df_lag[["lag_16", "lag_21", "lag_28"]].isna().sum())

for col in ["lag_16", "lag_21", "lag_28"]:
    train_df_lag[col] = train_df_lag[col].fillna(0)
    test_df_lag[col] = test_df_lag[col].fillna(0)

feature_cols_lag = feature_cols + ["lag_16", "lag_21", "lag_28"]

In [ ]:
cutoff_date = train_df_lag["date"].max() - pd.Timedelta(days=365)
train_df_small = train_df_lag[train_df_lag["date"] >= cutoff_date].copy()

last_date = train_df_small["date"].max()
val_start_date = last_date - pd.Timedelta(days=15)

train_part = train_df_small[train_df_small["date"] < val_start_date].copy()
valid_part = train_df_small[train_df_small["date"] >= val_start_date].copy()

print("train_part shape:", train_part.shape)
print("valid_part shape:", valid_part.shape)

X_train = train_part[feature_cols_lag].copy()
X_valid = valid_part[feature_cols_lag].copy()
X_test = test_df_lag[feature_cols_lag].copy()

y_train = np.log1p(train_part[target_col].values)
y_valid = np.log1p(valid_part[target_col].values)

model = CatBoostRegressor(
    iterations=300,
    learning_rate=0.05,
    depth=6,
    loss_function="RMSE",
    eval_metric="RMSE",
    random_seed=42,
    verbose=50
)

model.fit(
    X_train,
    y_train,
    cat_features=cat_cols,
    eval_set=(X_valid, y_valid),
    use_best_model=True,
    early_stopping_rounds=50
)

In [ ]:
def rmsle(y_true, y_pred):
    y_pred = np.clip(y_pred, 0, None)
    return np.sqrt(np.mean((np.log1p(y_pred) - np.log1p(y_true)) ** 2))

valid_pred_log = model.predict(X_valid)
valid_pred = np.expm1(valid_pred_log)
valid_pred = np.clip(valid_pred, 0, None)

valid_true = valid_part[target_col].values
score = rmsle(valid_true, valid_pred)
print("Validation RMSLE:", score)

In [ ]:
test_pred_log = model.predict(X_test)
test_pred = np.expm1(test_pred_log)
test_pred = np.clip(test_pred, 0, None)

submission = pd.DataFrame({
    "id": test_df["id"],
    "sales": test_pred
})

submission.to_csv("submission_safe_lag.csv", index=False)
print(submission.head())
print("submission_safe_lag.csv saved!")

## 当前结果
- experiment_name: `safe_lag_16_21_28_catboost_recent365`
- features_used: `stores + oil + basic_date_features + lag_16 + lag_21 + lag_28`
- validation_split: `last_16_days`
- model: `CatBoostRegressor`
- params: `iterations=300, depth=6, lr=0.05`
- score: `0.447398`

## 当前结论
lag 特征非常有效；safe lag（lag_16、lag_21、lag_28）是当前最佳版本。